In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BottleneckBlock(nn.Module):
    def __init__(self, channels: int, ratio: int = 4, dropout: float = 0.2) -> None:
        super().__init__()
        mid = max(channels // ratio, 16)
        self.block = nn.Sequential(
            nn.Conv1d(channels, mid, 1, bias=False),
            nn.GroupNorm(max(1, mid // 8), mid),
            nn.GELU(),
            nn.Conv1d(mid, mid, 3, padding=1, bias=False),
            nn.GroupNorm(max(1, mid // 8), mid),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(mid, channels, 1, bias=False),
            nn.GroupNorm(max(1, channels // 8), channels),
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))



class BlobNet(nn.Module):
    def __init__(self, n_max: int = N_MAX_BLOBS) -> None:
        super().__init__()
        self.n_max = n_max
 

        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.GroupNorm(4, 32),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
        )
 
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.GroupNorm(4, 64),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(3, 2)),
        )
 
       
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.GroupNorm(4, 128),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(5, 1)),  
        )
 
    
        self.skip1_pool = nn.AdaptiveMaxPool2d((1, 160))
        self.skip1_proj = nn.Sequential(
            nn.Conv1d(32, 128, 1, bias=False),
            nn.GroupNorm(4, 128),
            nn.GELU(),
        )
 
       
        self.skip2_pool = nn.AdaptiveMaxPool2d((1, 160))
        self.skip2_proj = nn.Sequential(
            nn.Conv1d(64, 128, 1, bias=False),
            nn.GroupNorm(4, 128),
            nn.GELU(),
        )
 
       
        self.body = nn.Sequential(
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-1 block
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-2 block
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-3 block A
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-3 block B
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-4 block A
            BottleneckBlock(128, ratio=4, dropout=0.2),   # level-4 block B
        )
 
       
        self.loc_refine = nn.Sequential(
            nn.Conv1d(128, 64, 3, padding=1),
            nn.GroupNorm(4, 64),
            nn.GELU(),
            nn.Conv1d(64, 32, 3, padding=1),
        )
        self.pool_pos = nn.AdaptiveAvgPool1d(n_max)
        self.head_pos = nn.Conv1d(32, 3, 1)
 
        
        self.pool_int = nn.AdaptiveMaxPool1d(n_max)
        self.head_int = nn.Conv1d(128, 1, 1)
 
    def forward(self, x: torch.Tensor):
        x = torch.asinh(x / 100.0)
 
        s1 = self.enc1(x)              
        s2 = self.enc2(s1)             
        feat = self.enc3(s2).squeeze(2)  
 
       
        skip1 = self.skip1_proj(self.skip1_pool(s1).squeeze(2))  
        skip2 = self.skip2_proj(self.skip2_pool(s2).squeeze(2))  
        feat = feat + skip1 + skip2                               
 
        feat = self.body(feat)           
 
       
        pos_feat = self.loc_refine(feat)  
        pos_feat = self.pool_pos(pos_feat)
        out_pos = self.head_pos(pos_feat)
 
       
        int_feat = self.pool_int(feat)  
        out_int = self.head_int(int_feat)
 
        x_pred = torch.sigmoid(out_pos[:, 0])
        y_pred = torch.sigmoid(out_pos[:, 1])
        exists_logit = out_pos[:, 2]
        int_norm_pred = torch.sigmoid(out_int[:, 0])
 
        return x_pred, y_pred, int_norm_pred, exists_logit
 
 